# React — Effects

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> This topic is mostly playground work. An Effect is behaviour over time in a real
> component, and a plain-JavaScript imitation would teach you the imitation.

## LESSON 39 — What an Effect is

LESSON 36 gave you React's one-line description of `useEffect` and told you to read it twice:

> connects a component to an external system

Not "runs code after render". That sentence is where every misuse of Effects begins, and this
lesson starts from the correct one.

### The distinction that matters: what caused it

You already have one way to make things happen — event handlers (topic 08). React draws the
line like this:

> **Effects let you specify side effects that are caused by rendering itself, rather than by
> a particular event.**

Read that as a question you ask about every piece of code: **what caused this to happen?**

| caused by | belongs in |
|---|---|
| the user clicked, typed, submitted | an **event handler** |
| this component appeared on screen, or its data changed | an **Effect** |

React's own example is the clearest one. In a chat app, *sending* a message is an event — a
particular person clicked a particular button. *Connecting to the server* is not: it has to
happen because the chat screen is on display, no matter how the user got there. One is caused
by an interaction, the other by rendering.

### What an Effect is for

> *Effects* let you run some code after rendering so that you can synchronize your component
> with some system outside of React.

**Outside React** is the operative phrase: a network connection, a browser API, a timer, a
third-party widget, something you have to set up and take down. If nothing outside React is
involved, you are probably not looking at an Effect — and LESSON 44 is entirely about that
mistake.

### When it runs

> **Effects run at the end of a commit after the screen updates.**

That sentence uses LESSON 38's vocabulary on purpose, and it is measurable. In the playground
experiment, each render and each Effect logs what the screen says *at that moment*. Pressing
`+1`:

```text
   render — count is 1, screen shows "0"
   render — count is 1, screen shows "0"
   effect — count is 1, screen shows "1"  <- already updated
```

During the render the component already knows `count` is `1`, but the screen still shows `0`
— nothing has been committed yet. By the time the Effect runs, the commit has happened and
the new number is on the page. That is the whole of "after the screen updates", observed.

> The two render lines are `<StrictMode>` (LESSON 3). Count pairs, not lines.

### The shape

**The React API.**

```jsx
import { useEffect, useState } from "react";

function Clock() {
  const [time, setTime] = useState("");

  useEffect(() => {
    document.title = time;      // reaching outside React
  });

  return <p>{time}</p>;
}
```

`useEffect` is a Hook, so LESSON 37 applies without exception: top level of the component,
never in a condition, a loop or a handler.

Written like that — with no second argument — the Effect runs **after every commit**. That is
almost never what you want, and it is the first of the three steps React lists for writing one:

> 1. **Declare an Effect.** By default, your Effect will run after every commit.
> 2. **Specify the Effect dependencies.** Most Effects should only re-run *when needed*.
> 3. **Add cleanup if needed.** "connect" needs "disconnect", "subscribe" needs
>    "unsubscribe", and "fetch" needs either "cancel" or "ignore".

This lesson is step 1 only. Step 2 is LESSON 40 and step 3 is LESSON 41, and an Effect is not
finished until you have considered all three.

### Do not reach for one by default

React says it plainly, and it is worth taking literally:

> **Don't rush to add Effects to your components.** … If your Effect only adjusts some state
> based on other state, you might not need an Effect.

You have already been taught the alternative without the warning attached: LESSON 29 said a
value you can calculate is not state. The same instinct applies here — if your Effect only
computes something from props or state, delete it and compute during render. LESSON 44 gives
this its own lesson, because it is the single most common way Effects are misused.

### Key Notes

- An Effect is for side effects **caused by rendering**, not by an event. Ask what caused it.
- Its job is synchronising with something **outside React** — network, browser API, timer,
  third-party code.
- Effects run at the **end of a commit, after the screen updates** — measurable, and the
  reason LESSON 38 came first.
- With no second argument an Effect runs after *every* commit. Dependencies are LESSON 40,
  cleanup is LESSON 41.

### Example

**In the playground.** There is no cell for this lesson — when an Effect runs relative to the
screen is real behaviour, and a JavaScript imitation would prove nothing.

Point `playground/src/App.jsx` at `./experiments/16-effects.jsx`, run it with the console
open, and press `+1`. Read the three lines in order and check the quoted screen values: the
renders still see the old number, the Effect sees the new one.

> The experiment deliberately reads the DOM during render to prove the timing. That is an
> impure render and you should never write it in real code — it is instrumentation, and the
> file says so.

### Exercise

**In the playground**, in `16-effects.jsx`, with the console open.

1. Press `+1` three times. How many `render` lines and how many `effect` lines appear per
   press? Explain the difference between the two counts in one sentence.
2. Press **set the same value (no change)**. Something surprising happens: the component logs
   renders but **no effect line appears at all**. Using LESSON 38's three steps, say which
   step did not happen and why that means no Effect ran.
3. Change the Effect to write the count into the document title:
   `document.title = \`count: ${count}\`;` — then watch the browser tab as you click. In a
   comment, say why this is a legitimate use of an Effect, naming the phrase from the lesson.
4. Now move that same line out of the Effect and into the `onClick` handler instead. It still
   works. In a comment, say which version you would keep and why — and what would break about
   the handler version if the count could also be changed by something other than that button.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each of these, say whether it belongs in an **event handler**, in an **Effect**, or in
**neither** — and in one line, why.

1. Sending a "message sent" analytics ping when the user clicks Send.
2. Opening a connection to a chat server while the chat screen is on display.
3. Calculating the total of a shopping basket to show under the list.
4. Setting `document.title` to the name of the product currently being viewed.
5. Starting a countdown timer when a quiz screen appears.
6. Showing a validation message after the user submits an invalid form.
7. Filtering a list of users by the text in a search box, to display the result.

Then answer: two of these are the same trap, and you met it in LESSON 29. Which two, and what
is the trap?

In [ ]:
// Your code here

## LESSON 40 — The dependency array

LESSON 39 left you with an Effect that runs after **every** commit:

```jsx
useEffect(() => {
  document.title = `count: ${count}`;
});
```

That is step 1 of three, and it is almost never what you want. Step 2 is telling React *when*
the Effect actually needs to run again.

### Three shapes, three behaviours

React's second argument to `useEffect` is a list of dependencies. There are exactly three
cases, and the difference between the last two is where people go wrong:

```jsx
useEffect(() => {
  // runs after every commit
});

useEffect(() => {
  // runs once when the component appears, and never again
}, []);

useEffect(() => {
  // runs when the component appears, and again whenever a or b has changed
}, [a, b]);
```

React's wording for the first and second:

> If you omit this argument, your Effect will re-run after every commit of the component.

> An Effect with empty dependencies doesn't re-run when any of your component's props or
> state change.

### What goes in the list

Not "whatever makes it work". React is specific:

> The list of all reactive values referenced inside of the `setup` code. Reactive values
> include props, state, and all the variables and functions declared directly inside your
> component body.

So the rule is mechanical: look at the code inside your Effect, find every value it reads that
comes from the component, and list those. Nothing else, and nothing missing.

There is one syntactic constraint worth knowing before you meet the error:

> The list of dependencies must have a constant number of items and be written inline like
> `[dep1, dep2, dep3]`.

You cannot build the array conditionally or hand it a variable — for the same reason Hooks
cannot be called conditionally (LESSON 37): React matches things up by position.

### How React decides "changed"

This is the whole lesson, and you already know the tool:

> React will compare each dependency with its previous value using the `Object.is` comparison.

`Object.is` — the same comparison from LESSON 25 that decides whether a state update is worth
re-rendering for. It compares **identity**, not contents. Two objects with identical fields are
two different values, and React will call that a change.

Measured in the playground experiment: clicking a room button that sets the room it is
**already** on produces no cleanup and no setup at all. The dependency did not change, so the
Effect did not re-run.

### The trap this creates

A primitive dependency behaves the way you expect: `"general"` is `"general"` on every render.

An **object, array or function created during render** does not. It is built fresh every time
the component runs, so it is a new value on every render, so `Object.is` says it changed, so
your Effect runs after every commit — exactly the thing the dependency array was supposed to
prevent:

```jsx
function ChatRoom({ roomId }) {
  const options = { roomId };          // a NEW object on every render

  useEffect(() => {
    connect(options);
  }, [options]);                        // so this runs every single time
}
```

The example cell below is that fact on its own, in plain JavaScript, with no React involved.

> The fix is not on this page. Sometimes you depend on the primitive instead of the object,
> sometimes you move the value inside the Effect, sometimes the object genuinely belongs
> outside the component. Working through those properly needs cleanup (LESSON 41) and the
> whole of LESSON 44, so for now the goal is to *recognise* the trap rather than to solve it.

### Key Notes

- No second argument → after every commit. `[]` → once when the component appears.
  `[a, b]` → when `a` or `b` changes.
- Dependencies are **every reactive value the Effect reads** — props, state, and values
  declared in the component body.
- React compares them with `Object.is`, which compares identity, not contents.
- An object, array or function created during render is a new value every render, so listing
  it as a dependency means the Effect never stops re-running.

### Example

**Runnable — plain JS.** No React here. The dependency array works entirely on the ordinary
JavaScript question *is this the same value as last time*, and that question has a surprising
answer often enough to be worth seeing on its own.

In [ ]:
// Values that look the same. Which ones ARE the same?
console.log("7 vs 7              ", Object.is(7, 7));
console.log("'general' vs same   ", Object.is("general", "general"));
console.log("{id:7} vs {id:7}    ", Object.is({ id: 7 }, { id: 7 }));
console.log("[1,2] vs [1,2]      ", Object.is([1, 2], [1, 2]));
console.log("() => {} vs same    ", Object.is(() => {}, () => {}));

// Now the same question, shaped like a component running twice with identical input.
function l40oneRender(roomId) {
  return {
    roomId,                   // came in as a prop
    options: { roomId },      // built fresh during this render
    onMessage: () => {},      // built fresh during this render
  };
}

const l40first = l40oneRender("general");
const l40second = l40oneRender("general");

console.log("");
console.log("roomId stable across renders?   ", Object.is(l40first.roomId, l40second.roomId));
console.log("options stable across renders?  ", Object.is(l40first.options, l40second.options));
console.log("onMessage stable across renders?", Object.is(l40first.onMessage, l40second.onMessage));

// Nothing about the room changed. Two of the three "dependencies" changed anyway.

That last block is the entire bug, written without React: the room is the same, the options
object is not, and `Object.is` only answers the second question.

### Exercise

**Part 1 — in the notebook.** Predict each answer before running it.

1. Write `l40changed(previous, next)` which takes two dependency **arrays** and returns `true`
   if React would re-run the Effect — that is, if any item differs by `Object.is`. Handle the
   first render, where there is no previous array, by returning `true`.
2. Check it against these pairs and log each result:
   - `["general", 3]` vs `["general", 3]`
   - `["general", 3]` vs `["random", 3]`
   - `[{ id: 1 }]` vs `[{ id: 1 }]`
   - `[]` vs `[]`
3. In a comment: the third pair looks like "nothing changed" to a human. Explain what React
   concludes and why, in one sentence. Then say what the fourth pair means for an Effect.

**Part 2 — in the playground**, in `17-cleanup.jsx`, with the console open. The Effect there
already has `[roomId]` as its dependency.

1. Click **random**, then click **random again**. How many times does the Effect re-run, and
   why is the second click different from the first?
2. Change the dependency array to `[]` and click the room buttons. What happens, and what is
   now wrong with what the screen says versus what the console says?
3. Put `[roomId]` back, then add a second dependency that is an object literal built in the
   component body — for example `const options = { roomId };` and `[roomId, options]`. Click
   anything at all, even a button that changes nothing about the room. What happens, and which
   line of the lesson predicted it?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each Effect, say what the dependency array **should** contain, and what goes wrong with
the one that is written.

```jsx
// 1
function Profile({ userId }) {
  useEffect(() => {
    console.log(`loading ${userId}`);
  }, []);
}

// 2
function Search({ query, pageSize }) {
  useEffect(() => {
    console.log(`searching ${query}, ${pageSize} per page`);
  }, [query]);
}

// 3
function Timer({ seconds }) {
  const config = { seconds };
  useEffect(() => {
    console.log(`starting ${config.seconds}s`);
  }, [config]);
}

// 4
function Title({ first, last }) {
  const fullName = `${first} ${last}`;
  useEffect(() => {
    document.title = fullName;
  }, [fullName]);
}
```

Then answer: number 4 lists a value built in the component body, exactly like number 3 — but
number 4 is fine. What is the difference, and which lesson's idea explains it?

In [ ]:
// Your code here

## LESSON 41 — Cleanup

Step 3 of React's three, and the one that separates an Effect that works from an Effect that
leaks:

> **Add cleanup if needed.** Some Effects need to specify how to stop, undo, or clean up
> whatever they were doing. For example, "connect" needs "disconnect", "subscribe" needs
> "unsubscribe", and "fetch" needs either "cancel" or "ignore".

### The shape

Your Effect can return a function, and that returned function is the cleanup:

```jsx
useEffect(() => {
  const connection = connect(roomId);      // setup

  return () => {
    connection.disconnect();               // cleanup
  };
}, [roomId]);
```

> Your setup function may also optionally return a *cleanup* function. The cleanup function
> should stop or undo whatever the setup is doing.

It is the only thing you may return from an Effect. Returning anything else is a mistake —
and it is how `async` Effects go wrong, which is LESSON 43.

### When React runs it

This is the part worth memorising, and React states it exactly:

> After every commit with changed dependencies, React will first run the cleanup function (if
> you provided it) with the **old** values, and then run your setup function with the **new**
> values. After your component is removed from the DOM, React will run your cleanup function.

Two occasions, then: **before re-synchronising**, and **on the way out**. Measured in the
playground, switching from the `general` room to `random`:

```text
   cleanup — disconnected from "general"
   setup   — connected to "random"
```

Cleanup first, and it still knows the old room. That is the "with the old values" clause doing
real work: the cleanup closes over the values from the render that set it up, which is exactly
what you need to undo it.

And unmounting the component:

```text
   cleanup — disconnected from "random"
```

Cleanup alone, no setup. The component is gone; there is nothing to synchronise with any more.

### Stop thinking "on mount" and "on unmount"

It is tempting to read `[]` as "on mount" and cleanup as "on unmount", and that reading breaks
the first time a dependency changes. React states the better model directly:

> Effects have a different lifecycle from components. Components may mount, update, or
> unmount. An Effect can only do two things: to start synchronizing something, and later to
> stop synchronizing it.

| | |
|---|---|
| **setup** | start synchronising with the outside thing |
| **cleanup** | stop synchronising with it |

An Effect does not have one lifetime tied to the component. It has as many
start-and-stop cycles as its dependencies demand, and the component appearing and disappearing
are only the first and last of them. React's own advice is to reason from the Effect's side:

> When you write and read Effects, think from each individual Effect's perspective (how to
> start and stop synchronization) rather than from the component's perspective (how it
> mounts, updates, or unmounts).

### Why development runs it twice

You were warned about this in LESSON 3, and here is the debt being paid. Mounting the chat in
the experiment logs:

```text
   setup   — connected to "random"
   cleanup — disconnected from "random"
   setup   — connected to "random"
```

Setup, cleanup, setup. React's explanation:

> When Strict Mode is on, React will **run one extra development-only setup+cleanup cycle**
> before the first real setup. This is a stress-test that ensures that your cleanup logic
> "mirrors" your setup logic and that it stops or undoes whatever the setup is doing.

It is not a bug and it is not something to switch off. It is React asking one question: *if I
run your setup twice, does your cleanup leave the world as it found it?* An Effect that opens
a connection without closing it fails that test loudly in development instead of quietly in
production, where the connections would pile up as the user moves around the app.

If the doubled logs bother you, the useful response is not to remove `<StrictMode>` — it is to
check that the cleanup really does undo the setup.

### Key Notes

- An Effect may return exactly one thing: a **cleanup function** that undoes its setup.
- React runs cleanup **before re-running the Effect** (with the old values) and **when the
  component is removed**.
- Think **start/stop synchronising**, not mount/unmount — an Effect can cycle many times
  while the component stays on screen.
- The development double-run is a deliberate stress-test of your cleanup, not a bug.

### Example

**In the playground.** No cell — the order in which setup and cleanup run is real behaviour
over time, and the console is the only honest place to watch it.

Point `playground/src/App.jsx` at `./experiments/17-cleanup.jsx`, run it, and open the console
before you click anything. Switch rooms and read the two lines in order; then unmount the chat
and read the single line.

### Exercise

**In the playground**, in `17-cleanup.jsx`.

1. With the console open, load the page. Write down the first three lines exactly, and say
   which of them is the "real" setup and which two are the development stress-test.
2. Switch from `general` to `random`. Two lines appear. Which value does the cleanup line
   print, and why is it the old one rather than the new one?
3. Unmount the chat. Then mount it again. Compare the two sets of lines and explain the
   difference in one sentence.
4. **Break it on purpose.** Delete the `return () => { … }` from the Effect, reload, and switch
   rooms four or five times. Nothing visibly breaks. In a comment, say what has actually
   happened and why you cannot see it — then say what you would expect to go wrong in a real
   app that opened a real connection.
5. Put the cleanup back. Now change the Effect so its cleanup logs a value the setup did *not*
   have — for instance move `const label = roomId.toUpperCase();` above the Effect and log
   `label` in the cleanup. Which room does it name when you switch, and what does that tell you
   about what a cleanup function can see?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Each Effect below needs cleanup and does not have it. For each, say what the cleanup function
should do — one line each, no code required.

```jsx
// 1
useEffect(() => {
  const id = setInterval(() => tick(), 1000);
}, []);

// 2
useEffect(() => {
  window.addEventListener("resize", handleResize);
}, []);

// 3
useEffect(() => {
  const socket = openSocket(roomId);
  socket.on("message", handleMessage);
}, [roomId]);

// 4
useEffect(() => {
  document.title = `Inbox (${unread})`;
}, [unread]);
```

Then answer:

- One of those four does **not** need a cleanup function. Which, and why is it different from
  the others?
- A colleague says the development double-run is "just noise" and wraps their app's
  `<StrictMode>` in a comment to silence it. Using number 1, describe concretely what they have
  chosen not to find out.

In [ ]:
// Your code here

## LESSON 42 — Debouncing with an Effect

Here is the first Effect in this topic that does something you would actually ship, and it is
built entirely from LESSON 41's cleanup.

### The problem

A search box whose Effect depends on the query fires once per keystroke:

```jsx
useEffect(() => {
  search(query);
}, [query]);
```

Type `react` and you have made five requests, four of which were obsolete before they were
answered. The user typed one search; your server received five.

### The fix: wait until the typing stops

**Debouncing** means starting a timer instead of acting immediately, and restarting that timer
every time the value changes. Only when the value stops changing for long enough does the
timer finish and the work happen.

In an Effect, that is startlingly little code, because the cancelling is just cleanup:

```jsx
useEffect(() => {
  const id = setTimeout(() => {
    search(query);
  }, 500);

  return () => clearTimeout(id);
}, [query]);
```

Read it against LESSON 41's rule — cleanup runs **before the Effect runs again** — and the
behaviour falls out:

1. you type `r` → the Effect schedules a search in 500ms
2. you type `e` → `query` changed, so React **cleans up first**: `clearTimeout` cancels the
   pending search. Then the Effect runs again and schedules a new one
3. …repeat for every keystroke…
4. you stop typing → nothing cancels the last timer, and 500ms later the search runs

Measured in the playground, typing `react` quickly:

```text
   scheduled — "r" in 500ms
   cancelled — "r"
   scheduled — "re" in 500ms
   cancelled — "re"
   scheduled — "rea" in 500ms
   cancelled — "rea"
   scheduled — "reac" in 500ms
   cancelled — "reac"
   scheduled — "react" in 500ms
   SEARCHING — "react"
```

Five keystrokes, four cancellations, **one search**. Type slowly and each one survives its
500ms, so each one runs — which is correct: those really were separate searches.

### Nothing here is new

That is the point worth pausing on. There is no debounce API, no library, no special Hook.
There is `setTimeout` and `clearTimeout` from the JavaScript course, plus the cleanup contract
from LESSON 41. The Effect's ability to cancel its own previous work is what makes debouncing
almost free.

> **A note on where this belongs.** Debouncing is the right tool for reducing *work outside
> React* — fewer network requests, fewer writes, fewer expensive calls. It is **not** the tool
> for making a slow list render faster; React has a purpose-built answer for that which you
> will meet in topic 23. React's own documentation makes the distinction: debouncing and
> throttling are still useful when "the work you're optimizing doesn't happen during
> rendering… they can let you fire fewer network requests."

> **On sourcing.** React's documentation does not publish a debounce-with-an-Effect recipe.
> The shape above is the standard construction, and every part of it is documented — the
> cleanup contract is React's, `setTimeout`/`clearTimeout` are the platform's. It is assembled
> here rather than quoted.

### Choosing the delay

There is no correct number. Too short and you have not saved anything; too long and the
interface feels broken. 300–500ms is the usual range for typing, and the honest way to pick is
to try it. What matters more is that the delay exists at all.

### Key Notes

- Debouncing = restart a timer on every change; act only when the changes stop.
- In an Effect it is `setTimeout` in the body and `clearTimeout` in the cleanup — LESSON 41
  doing the cancelling for you.
- Use it to reduce work **outside** React, above all network requests.
- Nothing about it is React-specific. The Effect just gives you a reliable place to cancel.

### Example

**In the playground.** No cell — debouncing is about timing, and a notebook cannot show you
five keystrokes racing a timer.

Point `playground/src/App.jsx` at `./experiments/18-debounce.jsx`, open the console, and type
a word quickly. Then clear the box and type another word slowly, pausing between letters.
Compare the two logs.

### Exercise

**In the playground**, in `18-debounce.jsx`.

1. Type a five-letter word as fast as you can. Count the `scheduled`, `cancelled` and
   `SEARCHING` lines. Then type another five-letter word with a clear pause after each letter.
   Explain the difference in one sentence.
2. Change `DELAY` to `2000`. Type something and wait. Then type something and *keep typing*
   every second or so, for ten seconds. What happens, and what does that tell you about
   choosing a delay?
3. Put `DELAY` back to `500`. Now **delete the `return () => { … }` line** and type a word
   quickly. How many searches run? Write down what the log looks like and explain, in terms of
   LESSON 41, exactly what stopped happening.
4. Put the cleanup back. In a comment: the Effect returns `clearTimeout(id)` wrapped in an
   arrow function. What would break if you wrote `return clearTimeout(id);` instead — and
   which lesson's rule does that violate?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each situation, say whether debouncing is the right tool, and why or why not.

1. A search box that queries the server as the user types.
2. A list of 5000 rows that re-renders slowly while the user types into a filter box.
3. Auto-saving a draft while the user writes a long comment.
4. A "load more" button that fetches the next page.
5. Sending each keystroke of a collaborative document to other users so they see live typing.

Then answer: two of these are about work that happens **during rendering** rather than outside
React. Which are they, and why does that make debouncing the wrong instinct — even though it
would appear to help?

In [ ]:
// Your code here

## LESSON 43 — Async work in an Effect

You know `fetch`, `async`/`await` and Promises from the JavaScript course. Putting them inside
an Effect introduces two problems that have nothing to do with fetching and everything to do
with time.

### The Effect callback cannot be `async`

This looks reasonable and is wrong:

```jsx
useEffect(async () => {          // ❌
  const data = await fetchTodos(userId);
  setTodos(data);
}, [userId]);
```

An `async` function always returns a Promise. LESSON 41 established that an Effect may return
exactly one thing — a cleanup function — so React receives a Promise where it expected a
function, and your cleanup silently does not exist.

The fix is to declare the async function **inside** the Effect and call it:

```jsx
useEffect(() => {
  async function startFetching() {
    const json = await fetchTodos(userId);
    setTodos(json);
  }

  startFetching();
}, [userId]);
```

The Effect itself stays synchronous and can still return a cleanup. Which it needs to, because
of the second problem.

### Responses do not arrive in the order you asked for

React's description of the bug is worth reading twice:

> Imagine you type `"hello"` fast. Then the `query` will change from `"h"`, to `"he"`,
> `"hel"`, `"hell"`, and `"hello"`. This will kick off separate fetches, but there is no
> guarantee about which order the responses will arrive in. For example, the `"hell"` response
> may arrive *after* the `"hello"` response. Since it will call `setResults()` last, you will
> be displaying the wrong search results. This is called a "race condition": two different
> requests "raced" against each other and came in a different order than you expected.

Nothing is broken in your code. Every request was correct, every response was correct, and the
last one to arrive wins — which happens to be the obsolete one.

Measured in the playground, with a fake API that answers short queries slowly. Typing `a` and
then `ab`:

```text
   request  — "a"
   request  — "ab"
   applied  — results for "ab"
   applied  — results for "a"      <- the stale one arrived last and overwrote
```

The search box reads `ab`. The results read `a`. That is the whole bug, and it is invisible
until your network is slow or your user types quickly.

### The fix: a flag the cleanup sets

> To fix the race condition, you need to add a cleanup function to ignore stale responses:

```jsx
useEffect(() => {
  let ignore = false;

  async function startFetching() {
    const json = await fetchTodos(userId);
    if (!ignore) {
      setTodos(json);
    }
  }

  startFetching();

  return () => {
    ignore = true;
  };
}, [userId]);
```

> This ensures that when your Effect fetches data, all responses except the last requested one
> will be ignored.

Every run of the Effect gets its **own** `ignore` variable, closed over by its own async
function — the same closure behaviour that let LESSON 41's cleanup remember the old room. When
the query changes, React cleans up first, which sets that particular run's flag to `true`. Its
response still arrives; it simply no longer does anything.

The same measurement with the flag on:

```text
   request  — "a"
   request  — "ab"
   applied  — results for "ab"
   ignored  — "a" (a newer request is in flight)
```

React is explicit that you cannot do better than this:

> You can't "undo" a network request that already happened, but your cleanup function should
> ensure that the fetch that's *not relevant anymore* does not keep affecting your application.

### Two requests in development is fine

You will see each request twice while developing, for the Strict Mode reason from LESSON 41.
React addresses it directly:

> In development, you will see two fetches in the Network tab. There is nothing wrong with
> that. With the approach above, the first Effect will immediately get cleaned up so its copy
> of the `ignore` variable will be set to `true`. So even though there is an extra request, it
> won't affect the state thanks to the `if (!ignore)` check.

> In production, there will only be one request.

### Key Notes

- Never make the Effect callback `async` — it would return a Promise where React expects a
  cleanup function. Declare an async function inside and call it.
- Responses can arrive out of order, so the **last to arrive wins** — and that may be the one
  you no longer want.
- A `let ignore = false` in the Effect, checked before setting state and set to `true` by the
  cleanup, discards stale responses.
- The doubled request in development is Strict Mode and is harmless once the flag is there.

### Example

**In the playground.** No cell — a race condition is a timing bug and needs two real requests
in flight.

Point `playground/src/App.jsx` at `./experiments/19-races.jsx`. The fake API deliberately
answers one-letter queries slowly, so the order is guaranteed rather than a matter of luck.
Type `a`, then quickly `ab`, and watch both the log and the result line.

### Exercise

**In the playground**, in `19-races.jsx`.

1. With **use the `ignore` flag** ticked, type `a` then quickly `ab`. Write down the four log
   lines in order and say which request was still in flight when the other finished.
2. Untick the box and repeat. Compare the result line with what is in the search box. Describe
   the bug as a user would report it — not in terms of Effects.
3. Tick the box again. In a comment, explain why each run of the Effect gets its *own* `ignore`
   variable rather than sharing one, and which earlier lesson explains that.
4. Change the delay function so **every** query takes the same time — `const delay = 300;`.
   Type quickly with the flag off. Does the bug still appear? Explain what that tells you about
   why race conditions are hard to catch.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

```jsx
// A
useEffect(async () => {
  const user = await getUser(userId);
  setUser(user);
}, [userId]);

// B
useEffect(() => {
  let ignore = false;
  getUser(userId).then((user) => setUser(user));
  return () => { ignore = true; };
}, [userId]);

// C
useEffect(() => {
  let ignore = false;
  async function load() {
    const user = await getUser(userId);
    if (!ignore) setUser(user);
  }
  load();
  return () => { ignore = true; };
}, [userId]);
```

1. Two of these are broken. Say which, and what exactly is wrong with each.
2. **B** is the interesting one: it has the flag, it has the cleanup, and it still does not
   work. Why not?
3. A colleague proposes fixing races by "just cancelling the request" instead. Using the quote
   from this lesson, say what is and is not possible there — and what the `ignore` flag is
   actually protecting.

In [ ]:
// Your code here

## LESSON 44 — You might not need an Effect

This lesson has been promised since LESSON 39 and it closes the topic, because the most
valuable thing you can know about Effects is when **not** to write one.

### What an Effect actually is

React's framing is blunter than most courses':

> Effects are an escape hatch from the React paradigm. They let you "step outside" of React and
> synchronize your components with some external system like a non-React widget, network, or
> the browser DOM. If there is no external system involved (for example, if you want to update
> a component's state when some props or state change), you shouldn't need an Effect. Removing
> unnecessary Effects will make your code easier to follow, faster to run, and less
> error-prone.

An **escape hatch**. Not a tool you reach for, a door you go through when you have to. Two
categories cover almost every unnecessary Effect ever written.

### 1. Do not use an Effect to transform data for rendering

> **You don't need Effects to transform data for rendering.** For example, let's say you want
> to filter a list before displaying it. You might feel tempted to write an Effect that updates
> a state variable when the list changes. However, this is inefficient. When you update the
> state, React will first call your component functions to calculate what should be on the
> screen. Then React will "commit" these changes to the DOM, updating the screen. Then React
> will run your Effects. If your Effect *also* immediately updates the state, this restarts the
> whole process from scratch! To avoid the unnecessary render passes, transform all the data at
> the top level of your components.

You already know this rule under a different name. LESSON 29 said a value you can calculate is
not state. Here is the same rule with the render/commit vocabulary from LESSON 38 attached, and
now you can see exactly what it costs:

```text
state changes -> render -> commit -> Effect -> sets state -> render -> commit -> …
```

Every derived value kept in state buys you a second trip round that loop, and a moment where
the screen shows the old total. The alternative is one line above the `return`.

### 2. Do not use an Effect to handle user events

> **You don't need Effects to handle user events.** … In the Buy button click event handler,
> you know exactly what happened. By the time an Effect runs, you don't know *what* the user
> did (for example, which button was clicked). This is why you'll usually handle user events in
> the corresponding event handlers.

This is LESSON 39's question — *what caused this?* — with the consequence spelled out. React's
example is the one to remember, because the bug is not subtle:

```jsx
useEffect(() => {
  // 🔴 Wrong: This Effect fires twice in development, exposing a problem in the code.
  fetch('/api/buy', { method: 'POST' });
}, []);
```

> You wouldn't want to buy the product twice. However, this is also why you shouldn't put this
> logic in an Effect. What if the user goes to another page and then presses Back? Your Effect
> would run again. You don't want to buy the product when the user *visits* a page; you want to
> buy it when the user *clicks* the Buy button.

> Buying is not caused by rendering; it's caused by a specific interaction. It should run only
> when the user presses the button.

And the sentence that reframes every doubled-Effect complaint you will ever hear:

> This illustrates that if remounting breaks the logic of your application, this usually
> uncovers existing bugs.

The development double-run did not create that bug. It found it.

### The rest of the list

React's guide covers twelve cases. The two above are the ones you will hit this week; the
others are worth knowing exist so you recognise them later:

| case | the short answer |
|---|---|
| Updating state based on props or state | calculate during render (LESSON 29) |
| Caching expensive calculations | `useMemo` — topic 23, and only after measuring |
| Resetting all state when a prop changes | give the component a different `key` |
| Adjusting some state when a prop changes | set it during rendering, not in an Effect |
| Sharing logic between event handlers | put the shared logic in a function both call |
| Sending a POST request | in the handler, unless it is caused by being on screen |
| Chains of computations | calculate what you can in one pass |
| Initializing the application | run it once outside your components |
| Notifying parent components | do it in the handler that caused the change |
| Subscribing to an external store | `useSyncExternalStore` exists for this |
| Fetching data | an Effect is legitimate — with cleanup (LESSON 43) |

Two of those are worth flagging now. **Fetching is on the list but is not forbidden** — React's
own conclusion is *"You don't need to move this fetch to an event handler"*; it needs cleanup,
which is why LESSON 43 came first. And **caching** points at `useMemo`, which this course
deliberately leaves until topic 23, after you have learned to measure.

### The test

Before writing an Effect, ask LESSON 39's question and then one more:

1. **What caused this?** An interaction → handler. The component being on screen → maybe an
   Effect.
2. **Is there a system outside React involved?** A network, a timer, the document title, a
   subscription, a third-party widget. If the answer is no — if the Effect only reads state and
   writes state — you are looking at a calculation, and it belongs above the `return`.

### Key Notes

- An Effect is an **escape hatch**, not a default. Removing unnecessary ones makes code
  "easier to follow, faster to run, and less error-prone".
- Transforming data for rendering is not an Effect's job — calculate during render.
- Handling user events is not an Effect's job — that is what handlers are for.
- If remounting breaks your app, the remount did not cause the bug; it revealed one.

### Example

**Runnable — plain JS.** The cost of the Effect-that-sets-state, counted rather than asserted.
No React here — this counts the passes each approach needs over the same data.

In [ ]:
// The same job done two ways: show only the unpaid invoices, and their total.
const l44invoices = [
  { id: "i1", client: "Baltic Foods", amount: 1200, paid: false },
  { id: "i2", client: "Northwind", amount: 800, paid: true },
  { id: "i3", client: "Ada Ltd", amount: 450, paid: false },
];

// --- Derived during render: one pass, no extra state, nothing to keep in step.
function l44derive(invoices) {
  const unpaid = invoices.filter((invoice) => !invoice.paid);
  const total = unpaid.reduce((sum, invoice) => sum + invoice.amount, 0);
  return { unpaid: unpaid.length, total };
}

console.log("derived:", JSON.stringify(l44derive(l44invoices)));

// --- The Effect version, counted. Each "pass" is a render+commit cycle.
let l44passes = 0;
let l44storedTotal = 0;

function l44renderWithEffect(invoices) {
  l44passes = l44passes + 1;                       // render
  const unpaid = invoices.filter((i) => !i.paid);  // still has to filter anyway

  // the "Effect" that runs after the commit and sets state:
  const nextTotal = unpaid.reduce((sum, i) => sum + i.amount, 0);
  if (nextTotal !== l44storedTotal) {
    l44storedTotal = nextTotal;
    l44renderWithEffect(invoices);                 // setting state renders again
  }
  return l44storedTotal;
}

console.log("effect version total:", l44renderWithEffect(l44invoices));
console.log("render passes needed:", l44passes, "<- one of them showed a stale total");

// The derived version needs one pass and cannot be stale. The Effect version needs two,
// and between them the screen briefly showed the OLD total.

The second pass is not a performance abstraction — it is a frame in which the user could see
the wrong number. That is the whole argument, and it is why this is a correctness lesson
rather than an optimisation one.

### Exercise

**Part 1 — in the notebook.** Each of these is an Effect that should not exist. Rewrite each
one as a single line that belongs above the `return`, and log the result.

```jsx
// A
const [fullName, setFullName] = useState("");
useEffect(() => {
  setFullName(first + " " + last);
}, [first, last]);

// B
const [visible, setVisible] = useState([]);
useEffect(() => {
  setVisible(items.filter((item) => item.category === category));
}, [items, category]);

// C
const [isEmpty, setIsEmpty] = useState(true);
useEffect(() => {
  setIsEmpty(items.length === 0);
}, [items]);
```

Write the three replacements as ordinary functions of their inputs, call them with sample data,
and log the results.

**Part 2 — judgement, in comments.** For each, say **handler**, **Effect**, or **neither**, and
give the one-line reason.

1. Logging an analytics event when a modal is opened by a button.
2. Logging an analytics event when a page becomes visible, however the user got there.
3. Recording the number of characters typed, to show under the textarea.
4. Saving a draft to the server 500ms after the user stops typing.
5. Sending a "mark as read" request when the user clicks a message.
6. Connecting to a websocket while a live-scores screen is open.
7. Sorting a table when the user clicks a column header.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

A colleague has this in a product page and reports that it "charges people twice in
development, but it's fine in production so we ship it":

```jsx
useEffect(() => {
  fetch("/api/buy", { method: "POST", body: JSON.stringify({ productId }) });
}, []);
```

Answer in comments:

1. Is the development double-run the bug? Answer using this lesson's sentence about remounting.
2. Describe a sequence of things a **real user in production** could do that charges them
   twice, without Strict Mode being involved at all.
3. Where does this code belong, and what single question from LESSON 39 would have placed it
   correctly the first time?
4. Your colleague's alternative fix is a module-level `let hasBought = false;` guard. Say why
   that is worse than moving the code, and what it would do on the user's *second* genuine
   purchase.

In [ ]:
// Your code here